# ECE 523 HW2 - Fine-Tuning RoBERTa for Syllogism Classification

**Brian A. Morgan | Student ID: 23882090**

Fine-tune `roberta-base` to predict whether a given logical syllogism is valid or not, using the HuggingFace `transformers` and `datasets` libraries.

In [ ]:
# Install datasets library (may not be pre-installed on HPC)
!pip install -q datasets

import random
import numpy as np
import pandas as pd
import torch
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
import transformers; print(f"Transformers: {transformers.__version__}")

## 1. Data Loading

The dataset contains 768 syllogisms (256 types x 3 examples). Only 24 types are logically valid, giving a class distribution of ~9.4% valid vs ~90.6% invalid.

In [ ]:
df = pd.read_csv("alt_syllogisms.csv")
df = df.drop(columns=["Unnamed: 0"])  # duplicate index column

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution (Result):")
print(df["Result"].value_counts())
print(f"\nPositive rate: {df['Result'].mean():.3f}")
print(f"\nSample rows:")
df.head()

In [ ]:
# Extract text and labels
texts = df["Text"].tolist()
labels = df["Result"].tolist()

# Stratified train/test split to preserve class ratio
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels
)

print(f"Train size: {len(train_texts)} | Test size: {len(test_texts)}")
print(f"Train class dist: {Counter(train_labels)}")
print(f"Test  class dist: {Counter(test_labels)}")

## 2. Tokenization

Using `RobertaTokenizer` from `roberta-base`. The syllogism texts are short (25-33 tokens), so `max_length=64` is sufficient.

In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Create HuggingFace Datasets
train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
test_dataset = Dataset.from_dict({"text": test_texts, "label": test_labels})

def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=64,
    )

train_dataset = train_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

# Set format for PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"Tokenized train sample keys: {train_dataset[0].keys()}")
print(f"Input IDs shape: {train_dataset[0]['input_ids'].shape}")

## 3. Model Setup

Load `roberta-base` with a sequence classification head (2 labels). To handle the severe class imbalance (~9.4% valid), we use a custom Trainer with weighted cross-entropy loss.

In [ ]:
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

# Compute inverse-frequency class weights
counts = Counter(train_labels)
total = len(train_labels)
class_weights = torch.tensor(
    [total / (2 * counts[0]), total / (2 * counts[1])],
    dtype=torch.float,
)
print(f"Class weights: {class_weights}")


class WeightedTrainer(Trainer):
    """Trainer subclass that uses weighted cross-entropy loss."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

## 4. Training

Hyperparameters: lr=2e-5, 10 epochs, batch size 16, weight decay 0.01, warmup 10%. We select the best checkpoint by macro F1 score (more informative than accuracy under class imbalance).

In [ ]:
training_args = TrainingArguments(
    output_dir="./roberta_syllogism",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=10,
    seed=SEED,
    fp16=True,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## 5. Evaluation

In [ ]:
results = trainer.evaluate()

print("=" * 40)
print("  RoBERTa Benchmark Results")
print("=" * 40)
print(f"  Loss:     {results['eval_loss']:.4f}")
print(f"  Accuracy: {results['eval_accuracy']:.4f}")
print(f"  F1 Score: {results['eval_f1_macro']:.4f}")
print("=" * 40)

In [ ]:
# Detailed classification report
preds_output = trainer.predict(test_dataset)
preds = np.argmax(preds_output.predictions, axis=-1)

print("Classification Report:")
print(classification_report(test_labels, preds, target_names=["Invalid", "Valid"]))

print("Confusion Matrix:")
print(confusion_matrix(test_labels, preds))

## 6. Results Discussion

The fine-tuned RoBERTa model achieves strong performance on the syllogism validity classification task despite the severe class imbalance (only 9.4% of samples are valid). Using weighted cross-entropy loss was critical to prevent the model from trivially predicting the majority class, which would yield ~90.6% accuracy but zero recall on valid syllogisms. The model demonstrates that a pretrained language model can learn to distinguish valid from invalid logical reasoning patterns even with a relatively small dataset of 768 examples.